# PADI Tabular — Demo: Compute SI P-value

Pipeline: Generate data → Train DeepSVDD → Detect anomaly → Compute SI p-value


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))

import numpy as np
import torch
from padi.PADI_tabular.si_padi import compute_p_value_one_vs_mean
from padi.PADI_tabular.gen_data import generate_data, CustomDataset
from padi.PADI_tabular.model import DeepSVDD


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_FEATURES = 10
N_TRAIN = 500
N_TEST = 100
N_REFS = 10
SIGMA = 1.0
DELTA = 3.0
ALPHA = 0.05

HIDDEN_DIMS = (32, 16, 8)
REPDIM = 16
N_EPOCHS_AE = 50
N_EPOCHS_SVDD = 100
BATCH_SIZE = 64
QUANTILE = 0.95

model_path = f'../model/deepsvdd_tabular_d{N_FEATURES}_r{REPDIM}.pth'
print(f'Device: {DEVICE}, model_path: {model_path}')


## 1. Build or Load Model

In [ ]:
if os.path.exists(model_path):
    print(f'Loading model from {model_path}...')
    model = DeepSVDD(device=DEVICE)
    model.load_model(model_path, device=DEVICE)
else:
    print('Training DeepSVDD from scratch...')

    X_train, y_train = generate_data(N_TRAIN, N_FEATURES, num_anomalies=0,
                                      delta=0, mu=0, sigma=SIGMA)
    train_set = CustomDataset(X_train, y_train)

    model = DeepSVDD(objective='one-class', device=DEVICE)
    model.set_network(in_features=N_FEATURES, repdim=REPDIM, hidden_dims=HIDDEN_DIMS)
    model.pretrain(train_set, n_epochs=N_EPOCHS_AE, batch_size=BATCH_SIZE, device=DEVICE)
    model.init_network_weights_from_pretraining()
    model.train(train_set, n_epochs=N_EPOCHS_SVDD, batch_size=BATCH_SIZE, device=DEVICE)
    model.compute_radius(train_set, quantile=QUANTILE, device=DEVICE)

    os.makedirs('../model', exist_ok=True)
    model.save_model(model_path)
    print(f'Model saved to {model_path}')

print(f'R² = {model.R_squared:.6f}')


## 2. Generate Test & Reference Data

In [ ]:
X_test_h0, _ = generate_data(N_TEST, N_FEATURES, num_anomalies=0,
                              delta=0, mu=0, sigma=SIGMA)
X_test_h1, y_test_h1 = generate_data(N_TEST, N_FEATURES, num_anomalies=10,
                                       delta=DELTA, mu=0, sigma=SIGMA)

# Normal reference pool (large), then sample N_REFS from it
X_ref_pool, _ = generate_data(N_TRAIN, N_FEATURES, num_anomalies=0,
                               delta=0, mu=0, sigma=SIGMA)
idx_ref = np.random.choice(len(X_ref_pool), N_REFS, replace=False)
X_refs = X_ref_pool[idx_ref]

print(f'Test H\u2080: {len(X_test_h0)}, Test H\u2081: {len(X_test_h1)}, Ref pool: {len(X_ref_pool)}, Refs sampled: {len(X_refs)}')


## 3. Detect Anomaly & Compute P-value

In [ ]:
center_c = model.c.cpu().numpy()
net = model.net.cpu()
net.eval()

# --- H₀: Normal data (expect p-value > α → do NOT reject) ---
print("=" * 60)
print("H₀: Testing on NORMAL data")
print("=" * 60)

X_tensor = torch.from_numpy(X_test_h0).float()
O_h0, scores_h0, _ = model.anomaly_detection_fixed_radius(X_tensor, model.R_squared)

for idx in O_h0:
    print(f"Sample {idx} (label=0): score={scores_h0[idx]:.6f} > R²={model.R_squared:.6f}")
    print(f"  → Detected as anomaly!")

    p_value = compute_p_value_one_vs_mean(
        X_test_h0[idx], X_refs, SIGMA, net,
        model.R_squared, center_c)

    print(f"  SI P-value: {p_value}")
    if p_value is not None:
        print(f"  Reject H₀ at α={ALPHA}? {'YES' if p_value < ALPHA else 'NO'}")
    break
else:
    print("No anomalies detected in H₀ data — FPR is zero (good!)")

# --- H₁: Anomalous data (expect p-value < α → reject) ---
print()
print("=" * 60)
print("H₁: Testing on ANOMALOUS data")
print("=" * 60)

X_tensor = torch.from_numpy(X_test_h1).float()
O_h1, scores_h1, _ = model.anomaly_detection_fixed_radius(X_tensor, model.R_squared)
true_detected = [i for i in O_h1 if y_test_h1[i] == 1]

for idx in true_detected:
    print(f"Sample {idx} (label=1): score={scores_h1[idx]:.6f} > R²={model.R_squared:.6f}")
    print(f"  → Detected as anomaly!")

    p_value = compute_p_value_one_vs_mean(
        X_test_h1[idx], X_refs, SIGMA, net,
        model.R_squared, center_c)

    print(f"  SI P-value: {p_value}")
    if p_value is not None:
        print(f"  Reject H₀ at α={ALPHA}? {'YES' if p_value < ALPHA else 'NO'}")
    break
else:
    print("No true anomalies detected — try increasing DELTA")
